# Этап 4: Чанкование

**Стратегия:**
1. Логическая нарезка по Markdown-заголовкам (`MarkdownHeaderTextSplitter`)
2. Рекурсивная нарезка длинных секций (`RecursiveCharacterTextSplitter`)
3. Фильтрация слишком коротких чанков (< 50 символов)

**Принцип разделения текста и метаданных:**
- `text` -- чистый контент чанка (только для эмбеддинга)
- `metadata` -- структурированный dict с заголовками секций + метаданные статьи из CSV
- Метаданные НЕ вклеиваются в текст, а сохраняются отдельно для Qdrant payload

In [8]:
from pathlib import Path
from datasets import Dataset
import pandas as pd
import os
import logging

from transformers import AutoTokenizer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [9]:
# --- КОНФИГ ---
MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
INPUT_FILE = Path('../../data/processed/parquet/processed_data.parquet')
METADATA_CSV = Path('../../data/metadata/arxiv_NLP_1991_2026_metadata.csv')
OUTPUT_PARQUET = Path('../../data/processed/all_chunks.parquet')

CHUNK_SIZE_TOKENS = 512
CHUNK_OVERLAP_TOKENS = 50
MIN_CHUNK_LENGTH = 50  # Минимальная длина чанка в символах

BATCH_SIZE = 100

## 1. Загрузка данных и метаданных

In [10]:
# --- Загрузка основного датасета ---
ds = Dataset.from_parquet(str(INPUT_FILE))
logger.info(f"Dataset loaded: {len(ds)} documents")

2026-02-13 00:00:37,033 - INFO - Dataset loaded: 16345 documents


In [11]:
# --- Загрузка метаданных статей из CSV ---
# Подтягиваем title, authors, published для обогащения payload в Qdrant
meta_df = pd.read_csv(
    METADATA_CSV,
    usecols=['arxiv_id', 'title', 'authors', 'published'],
    dtype={'arxiv_id': str, 'title': str, 'authors': str, 'published': str}
)

# Строим lookup: doc_id -> {title, authors, published}
paper_metadata = (
    meta_df
    .dropna(subset=['arxiv_id'])
    .set_index('arxiv_id')
    [['title', 'authors', 'published']]
    .to_dict(orient='index')
)

logger.info(f"Paper metadata loaded: {len(paper_metadata)} entries")

# Проверка пересечения
doc_ids_in_ds = set(ds['doc_id'])
doc_ids_in_meta = set(paper_metadata.keys())
coverage = len(doc_ids_in_ds & doc_ids_in_meta) / len(doc_ids_in_ds) * 100
logger.info(f"Metadata coverage: {coverage:.1f}% of documents have metadata")

2026-02-13 00:00:38,156 - INFO - Paper metadata loaded: 77379 entries
2026-02-13 00:00:38,396 - INFO - Metadata coverage: 100.0% of documents have metadata


## 2. Инициализация сплиттеров

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3")
    ]
)

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=CHUNK_SIZE_TOKENS,
    chunk_overlap=CHUNK_OVERLAP_TOKENS
)

logger.info(f"Splitters ready: chunk_size={CHUNK_SIZE_TOKENS} tokens, overlap={CHUNK_OVERLAP_TOKENS} tokens")

2026-02-13 00:00:39,034 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-13 00:00:39,218 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/config.json "HTTP/1.1 200 OK"
2026-02-13 00:00:39,426 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-13 00:00:39,609 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/tokenizer_config.json "HTTP/1.1 200 OK"
2026-02-13 00:00:39,820 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-Embedding-0.6B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-02-13 00:00:40,030 - INFO - HTTP Request: GET https://huggin

## 3. Функция чанкования

**Ключевое отличие от предыдущей версии:**
- `text` содержит ТОЛЬКО чистый контент (`chunk.page_content`)
- Заголовки секций сохраняются в `metadata` как отдельные поля
- Метаданные статьи (title, authors, published) подтягиваются из CSV
- Всё это уедет в Qdrant payload, а НЕ в эмбеддинг

In [13]:
def process_batch(batch, md_splitter, text_splitter, paper_metadata):
    """Чанкование батча документов с разделением текста и метаданных.

    Текст чанка -- чистый контент без заголовков.
    Метаданные (заголовки секций + info о статье) хранятся отдельно.

    Все зависимости определены внутри функции,
    т.к. num_proc > 1 порождает отдельные процессы без доступа к глобалам Jupyter.

    Returns:
        dict с ключами: text, doc_id, metadata
    """
    import logging
    logger = logging.getLogger(__name__)

    MIN_CHUNK_LENGTH = 50

    chunk_texts = []
    chunk_doc_ids = []
    chunk_metadata_list = []

    for doc_id, md_content in zip(batch['doc_id'], batch['md']):
        # Пропускаем невалидные документы
        if not md_content or not isinstance(md_content, str) or len(md_content.strip()) == 0:
            continue

        try:
            # 1. Нарезка по Markdown-заголовкам
            sections = md_splitter.split_text(md_content)

            # 2. Рекурсивная нарезка длинных секций
            chunks = text_splitter.split_documents(sections)

            # 3. Метаданные статьи из CSV (title, authors, published)
            paper_meta = paper_metadata.get(doc_id, {})

            for chunk_idx, chunk in enumerate(chunks):
                text = chunk.page_content.strip()

                # Фильтрация слишком коротких чанков
                if len(text) < MIN_CHUNK_LENGTH:
                    continue

                # Метаданные: заголовки секций из MarkdownHeaderTextSplitter
                header_1 = chunk.metadata.get('header_1', '')
                header_2 = chunk.metadata.get('header_2', '')
                header_3 = chunk.metadata.get('header_3', '')

                # Собираем metadata dict
                meta = {
                    # Заголовки секций (структура документа)
                    'header_1': header_1,
                    'header_2': header_2,
                    'header_3': header_3,
                    # Метаданные статьи из arXiv API
                    'title': paper_meta.get('title', ''),
                    'authors': paper_meta.get('authors', ''),
                    'published': paper_meta.get('published', ''),
                    # Позиция чанка в документе
                    'chunk_index': chunk_idx,
                }

                chunk_texts.append(text)
                chunk_doc_ids.append(doc_id)
                chunk_metadata_list.append(meta)

        except Exception as e:
            logger.error(f"Error processing doc_id {doc_id}: {e}")
            continue

    return {
        'text': chunk_texts,
        'doc_id': chunk_doc_ids,
        'metadata': chunk_metadata_list,
    }

## 4. Запуск чанкования

In [14]:
chunked_ds = ds.map(
    process_batch,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=ds.column_names,
    num_proc=os.cpu_count(),
    fn_kwargs={
        'md_splitter': md_splitter,
        'text_splitter': text_splitter,
        'paper_metadata': paper_metadata,
    }
)

logger.info(f"Chunking complete: {len(chunked_ds)} chunks")

Map (num_proc=12):   0%|          | 0/16345 [00:00<?, ? examples/s]

2026-02-13 00:07:21,756 - INFO - Chunking complete: 652651 chunks


## 5. Проверка результатов

In [15]:
# Выборочная проверка: текст должен быть чистым, метаданные -- отдельно
sample = chunked_ds[0]
print('=== SAMPLE CHUNK ===')
print(f"doc_id:   {sample['doc_id']}")
print(f"text:     {sample['text'][:200]}...")
print(f"metadata: {sample['metadata']}")
print()

# Статистика
text_lengths = [len(t) for t in chunked_ds['text']]
print(f"Total chunks:      {len(chunked_ds)}")
print(f"Avg chunk length:  {sum(text_lengths) / len(text_lengths):.0f} chars")
print(f"Min chunk length:  {min(text_lengths)} chars")
print(f"Max chunk length:  {max(text_lengths)} chars")

# Проверка, что title подтянулся
titles_filled = sum(1 for m in chunked_ds['metadata'] if m.get('title'))
print(f"Chunks with title: {titles_filled}/{len(chunked_ds)} ({titles_filled/len(chunked_ds)*100:.1f}%)")

=== SAMPLE CHUNK ===
doc_id:   2501.00691v2
text:     ###### Abstract  
Large language models (LLMs) have revolutionised many fields, with LLM-as-a-service (LLMSaaS) offering accessible, general-purpose solutions without costly task-specific training. In...
metadata: {'authors': 'Md Rakibul Hasan, Yue Yao, Md Zakir Hossain, Aneesh Krishna, Imre Rudas, Shafin Rahman, Tom Gedeon', 'chunk_index': 1, 'header_1': 'Labels Generated by Large Language Models Help Measure People’s Empathy in Vitro', 'header_2': '', 'header_3': '', 'published': '2025-01-01', 'title': "Labels Generated by Large Language Models Help Measure People's Empathy in Vitro"}

Total chunks:      652651
Avg chunk length:  1245 chars
Min chunk length:  50 chars
Max chunk length:  7099 chars
Chunks with title: 652651/652651 (100.0%)


## 6. Сохранение

In [16]:
OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
chunked_ds.to_parquet(str(OUTPUT_PARQUET))

logger.info(f"Saved to {OUTPUT_PARQUET}: {len(chunked_ds)} chunks")

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

2026-02-13 00:07:56,226 - INFO - Saved to ..\..\data\processed\all_chunks.parquet: 652651 chunks
